# Histogram and Sorting

Master atomic operations, build privatized histograms for high throughput, and explore GPU sorting algorithms including bitonic sort, merge sort, and radix sort.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-histogram-sort)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: Atomic Operations

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
import time

# ============================================================
# Demonstrate race condition vs atomic operation
# ============================================================

@cuda.jit
def broken_increment(result):
    """WRONG: Race condition -- many updates are lost."""
    result[0] = result[0] + 1  # Non-atomic read-modify-write

@cuda.jit
def correct_increment(result):
    """CORRECT: Atomic add ensures all updates are counted."""
    cuda.atomic.add(result, 0, 1)

print("=" * 60)
print("Race Condition Demonstration")
print("=" * 60)

n_threads = 1024
n_blocks = 10
total_expected = n_threads * n_blocks

# Broken version (race condition)
result_broken = cuda.to_device(np.array([0], dtype=np.int32))
broken_increment[n_blocks, n_threads](result_broken)
cuda.synchronize()
broken_val = result_broken.copy_to_host()[0]

# Correct version (atomic)
result_correct = cuda.to_device(np.array([0], dtype=np.int32))
correct_increment[n_blocks, n_threads](result_correct)
cuda.synchronize()
correct_val = result_correct.copy_to_host()[0]

print(f"Expected:     {total_expected}")
print(f"Non-atomic:   {broken_val} (lost {total_expected - broken_val} updates!)")
print(f"Atomic:       {correct_val}")

# ============================================================
# Atomic add for simple histogram
# ============================================================
@cuda.jit
def naive_histogram(data, hist, n):
    """Simple histogram using atomic add."""
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if tid < n:
        bin_idx = int(data[tid])
        cuda.atomic.add(hist, bin_idx, 1)

print("\n" + "=" * 60)
print("Simple Histogram with Atomics")
print("=" * 60)

# Create data: random integers 0-9
np.random.seed(42)
data = np.random.randint(0, 10, size=10000).astype(np.int32)
d_data = cuda.to_device(data)
d_hist = cuda.to_device(np.zeros(10, dtype=np.int32))

threads = 256
blocks = (len(data) + threads - 1) // threads
naive_histogram[blocks, threads](d_data, d_hist, len(data))

hist_gpu = d_hist.copy_to_host()
hist_cpu = np.bincount(data, minlength=10)

print(f"GPU histogram: {hist_gpu}")
print(f"CPU histogram: {hist_cpu}")
print(f"Match: {np.array_equal(hist_gpu, hist_cpu)}")

# ============================================================
# Contention benchmark
# ============================================================
@cuda.jit
def high_contention_kernel(result, n):
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if tid < n:
        cuda.atomic.add(result, 0, 1)  # All threads -> 1 address

@cuda.jit
def low_contention_kernel(result, n):
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if tid < n:
        cuda.atomic.add(result, tid % 256, 1)  # Spread across 256

print("\n" + "=" * 60)
print("Contention Benchmark (100,000 atomic adds)")
print("=" * 60)

n = 100000
threads = 256
blocks = (n + threads - 1) // threads

# High contention
result_high = cuda.to_device(np.zeros(1, dtype=np.int32))
cuda.synchronize()
start = time.perf_counter()
for _ in range(100):
    result_high = cuda.to_device(np.zeros(1, dtype=np.int32))
    high_contention_kernel[blocks, threads](result_high, n)
cuda.synchronize()
high_time = (time.perf_counter() - start) / 100

# Low contention
result_low = cuda.to_device(np.zeros(256, dtype=np.int32))
cuda.synchronize()
start = time.perf_counter()
for _ in range(100):
    result_low = cuda.to_device(np.zeros(256, dtype=np.int32))
    low_contention_kernel[blocks, threads](result_low, n)
cuda.synchronize()
low_time = (time.perf_counter() - start) / 100

print(f"High contention (1 addr):   {high_time*1e6:.0f} us")
print(f"Low contention (256 addrs): {low_time*1e6:.0f} us")
print(f"Speedup from reducing contention: {high_time/low_time:.1f}x")

### Lesson example: Parallel Histogram

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
import time

# ============================================================
# Approach 1: Naive global-memory histogram
# ============================================================
@cuda.jit
def naive_histogram_kernel(data, hist, n):
    """Each thread atomically increments a global histogram bin."""
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if tid < n:
        bin_idx = int(data[tid])
        cuda.atomic.add(hist, bin_idx, 1)

# ============================================================
# Approach 2: Privatized histogram with shared memory
# ============================================================
@cuda.jit
def privatized_histogram_kernel(data, hist, n, num_bins):
    """Per-block private histogram in shared memory, then merge."""
    shared_hist = cuda.shared.array(256, dtype=numba.int32)
    tid = cuda.threadIdx.x
    global_tid = tid + cuda.blockIdx.x * cuda.blockDim.x
    
    # Initialize shared histogram
    if tid < num_bins:
        shared_hist[tid] = 0
    cuda.syncthreads()
    
    # Accumulate into shared histogram (fast shared-memory atomics)
    if global_tid < n:
        bin_idx = int(data[global_tid])
        cuda.atomic.add(shared_hist, bin_idx, 1)
    cuda.syncthreads()
    
    # Merge shared -> global (only num_bins threads needed)
    if tid < num_bins:
        cuda.atomic.add(hist, tid, shared_hist[tid])

# ============================================================
# Approach 3: Privatized + coarsened
# ============================================================
@cuda.jit
def coarsened_histogram_kernel(data, hist, n, num_bins):
    """Each thread processes 4 elements (coarsening factor = 4)."""
    shared_hist = cuda.shared.array(256, dtype=numba.int32)
    tid = cuda.threadIdx.x
    
    # Initialize
    if tid < num_bins:
        shared_hist[tid] = 0
    cuda.syncthreads()
    
    # Each thread processes 4 elements
    coarsen = 4
    base = (cuda.blockIdx.x * cuda.blockDim.x + tid) * coarsen
    for i in range(coarsen):
        idx = base + i
        if idx < n:
            bin_idx = int(data[idx])
            cuda.atomic.add(shared_hist, bin_idx, 1)
    cuda.syncthreads()
    
    # Merge
    if tid < num_bins:
        cuda.atomic.add(hist, tid, shared_hist[tid])

# ============================================================
# Test correctness
# ============================================================
print("=" * 60)
print("Histogram Correctness Test")
print("=" * 60)

np.random.seed(42)
n = 1_000_000
num_bins = 256
data = np.random.randint(0, num_bins, size=n).astype(np.int32)
d_data = cuda.to_device(data)
expected = np.bincount(data, minlength=num_bins).astype(np.int32)

threads = 256
blocks = (n + threads - 1) // threads

for name, kernel, coarsened in [
    ("Naive global", naive_histogram_kernel, False),
    ("Privatized", privatized_histogram_kernel, False),
    ("Coarsened", coarsened_histogram_kernel, True),
]:
    d_hist = cuda.to_device(np.zeros(num_bins, dtype=np.int32))
    if coarsened:
        coarsen_blocks = (n + threads * 4 - 1) // (threads * 4)
        kernel[coarsen_blocks, threads](d_data, d_hist, n, num_bins)
    elif "Privatized" in name:
        kernel[blocks, threads](d_data, d_hist, n, num_bins)
    else:
        kernel[blocks, threads](d_data, d_hist, n)
    
    hist_gpu = d_hist.copy_to_host()
    match = np.array_equal(hist_gpu, expected)
    print(f"  {name:20s}: {'PASS' if match else 'FAIL'}")
    if not match:
        print(f"    Max diff: {np.max(np.abs(hist_gpu - expected))}")

# ============================================================
# Performance comparison
# ============================================================
print("\n" + "=" * 60)
print(f"Performance Benchmark ({n:,} elements, {num_bins} bins)")
print("=" * 60)

iters = 50

# Warm up
d_hist = cuda.to_device(np.zeros(num_bins, dtype=np.int32))
naive_histogram_kernel[blocks, threads](d_data, d_hist, n)
cuda.synchronize()

# Naive
start = time.perf_counter()
for _ in range(iters):
    d_hist = cuda.to_device(np.zeros(num_bins, dtype=np.int32))
    naive_histogram_kernel[blocks, threads](d_data, d_hist, n)
cuda.synchronize()
naive_time = (time.perf_counter() - start) / iters

# Privatized
start = time.perf_counter()
for _ in range(iters):
    d_hist = cuda.to_device(np.zeros(num_bins, dtype=np.int32))
    privatized_histogram_kernel[blocks, threads](d_data, d_hist, n, num_bins)
cuda.synchronize()
priv_time = (time.perf_counter() - start) / iters

# Coarsened
coarsen_blocks = (n + threads * 4 - 1) // (threads * 4)
start = time.perf_counter()
for _ in range(iters):
    d_hist = cuda.to_device(np.zeros(num_bins, dtype=np.int32))
    coarsened_histogram_kernel[coarsen_blocks, threads](d_data, d_hist, n, num_bins)
cuda.synchronize()
coarse_time = (time.perf_counter() - start) / iters

data_size_gb = n * 4 / 1e9  # int32 = 4 bytes
print(f"  Naive:      {naive_time*1e3:.2f} ms  ({data_size_gb/naive_time:.1f} GB/s)")
print(f"  Privatized: {priv_time*1e3:.2f} ms  ({data_size_gb/priv_time:.1f} GB/s)")
print(f"  Coarsened:  {coarse_time*1e3:.2f} ms  ({data_size_gb/coarse_time:.1f} GB/s)")
print(f"  Speedup (privatized vs naive): {naive_time/priv_time:.1f}x")
print(f"  Speedup (coarsened vs naive):  {naive_time/coarse_time:.1f}x")

### Lesson example: GPU Sorting

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
import time
import math

# ============================================================
# Bitonic Sort (GPU, power-of-2 sizes)
# ============================================================
@cuda.jit
def bitonic_sort_step_kernel(data, j, k, n):
    """One step of bitonic sort network.
    
    j: distance between compared elements
    k: determines sort direction (ascending/descending for sub-sequences)
    """
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if tid >= n:
        return
    
    # XOR to find partner
    partner = tid ^ j
    
    if partner > tid and partner < n:
        if (tid & k) == 0:  # Ascending
            if data[tid] > data[partner]:
                temp = data[tid]
                data[tid] = data[partner]
                data[partner] = temp
        else:  # Descending
            if data[tid] < data[partner]:
                temp = data[tid]
                data[tid] = data[partner]
                data[partner] = temp


def bitonic_sort(data):
    """Host function: runs bitonic sort on GPU."""
    n = len(data)
    # Pad to power of 2
    padded_n = 1 << (n - 1).bit_length()
    padded = np.full(padded_n, np.iinfo(np.int32).max, dtype=np.int32)
    padded[:n] = data
    
    d_data = cuda.to_device(padded)
    threads = 256
    blocks = (padded_n + threads - 1) // threads
    
    k = 2
    while k <= padded_n:
        j = k >> 1
        while j >= 1:
            bitonic_sort_step_kernel[blocks, threads](d_data, j, k, padded_n)
            cuda.synchronize()
            j >>= 1
        k <<= 1
    
    return d_data.copy_to_host()[:n]


# ============================================================
# Radix Sort (conceptual, using NumPy scan)
# ============================================================
def radix_sort_cpu(data):
    """Radix sort using scan primitive (CPU reference).
    
    Demonstrates the algorithm that GPUs implement with parallel scan.
    """
    n = len(data)
    output = data.copy().astype(np.int32)
    
    for bit in range(32):  # 32 bits for int32
        bits = (output >> bit) & 1  # Extract bit
        zeros = 1 - bits
        
        # Exclusive scan of zeros (positions for 0-bit elements)
        zero_scan = np.concatenate([[0], np.cumsum(zeros)[:-1]])
        total_zeros = int(np.sum(zeros))
        
        # Exclusive scan of bits (positions for 1-bit elements)
        one_scan = np.concatenate([[0], np.cumsum(bits)[:-1]])
        
        # Scatter
        new_output = np.empty_like(output)
        for i in range(n):
            if bits[i] == 0:
                new_output[zero_scan[i]] = output[i]
            else:
                new_output[total_zeros + one_scan[i]] = output[i]
        output = new_output
    
    return output


# ============================================================
# Test correctness
# ============================================================
print("=" * 60)
print("Sorting Algorithm Tests")
print("=" * 60)

# Bitonic sort
np.random.seed(42)
for n in [8, 16, 100, 1000]:
    data = np.random.randint(0, 10000, size=n).astype(np.int32)
    sorted_gpu = bitonic_sort(data)
    expected = np.sort(data)
    match = np.array_equal(sorted_gpu, expected)
    print(f"  Bitonic sort N={n:>5}: {'PASS' if match else 'FAIL'}")

# Radix sort
print()
for n in [8, 100, 1000, 5000]:
    data = np.random.randint(0, 10000, size=n).astype(np.int32)
    sorted_radix = radix_sort_cpu(data)
    expected = np.sort(data)
    match = np.array_equal(sorted_radix, expected)
    print(f"  Radix sort  N={n:>5}: {'PASS' if match else 'FAIL'}")

# ============================================================
# Benchmark bitonic sort
# ============================================================
print("\n" + "=" * 60)
print("Bitonic Sort Performance")
print("=" * 60)

for n in [1024, 4096, 16384, 65536]:
    data = np.random.randint(0, 100000, size=n).astype(np.int32)
    
    # GPU bitonic sort
    _ = bitonic_sort(data)  # warm up
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(10):
        _ = bitonic_sort(data)
    cuda.synchronize()
    gpu_time = (time.perf_counter() - start) / 10
    
    # CPU numpy sort
    start = time.perf_counter()
    for _ in range(10):
        _ = np.sort(data)
    cpu_time = (time.perf_counter() - start) / 10
    
    print(f"  N={n:>6}: GPU={gpu_time*1e3:.2f}ms, CPU={cpu_time*1e3:.2f}ms, "
          f"Speedup={cpu_time/gpu_time:.2f}x")

print("\nNote: Our simple bitonic sort has high kernel launch overhead.")
print("Production GPU sorts (CUB/Thrust) fuse steps and are 10-100x faster.")

---

## Exercise: Privatized Histogram for Image Intensity Distribution


In [ ]:
import numpy as np
from numba import cuda
import numba
import time

NUM_BINS = 256

# ============================================================
# TODO 1: Naive histogram (global atomics)
# ============================================================
@cuda.jit
def naive_histogram_kernel(data, hist, n):
    """Each thread atomically increments a global histogram bin."""
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    # TODO: If tid < n, get bin index from data[tid], atomic add to hist
    pass


# ============================================================
# TODO 2: Privatized histogram (shared memory)
# ============================================================
@cuda.jit
def privatized_histogram_kernel(data, hist, n, num_bins):
    """Per-block private histogram in shared memory."""
    shared_hist = cuda.shared.array(256, dtype=numba.int32)
    tid = cuda.threadIdx.x
    global_tid = tid + cuda.blockIdx.x * cuda.blockDim.x
    
    # TODO: Initialize shared_hist to zeros (threads with tid < num_bins)
    # TODO: syncthreads()
    # TODO: Accumulate into shared_hist using atomic add
    # TODO: syncthreads()
    # TODO: Merge shared_hist into global hist
    pass


# ============================================================
# TODO 3: Coarsened privatized histogram
# ============================================================
@cuda.jit
def coarsened_histogram_kernel(data, hist, n, num_bins):
    """Each thread processes 4 elements."""
    shared_hist = cuda.shared.array(256, dtype=numba.int32)
    tid = cuda.threadIdx.x
    COARSEN = 4
    
    # TODO: Initialize shared_hist
    # TODO: Each thread processes COARSEN elements
    # TODO: Merge into global histogram
    pass


# ============================================================
# Testing and Benchmarking
# ============================================================
np.random.seed(42)
n = 2_000_000
# Simulate grayscale image (biased toward middle intensities)
data = np.clip(
    np.random.normal(128, 40, size=n).astype(np.int32),
    0, 255
).astype(np.int32)
d_data = cuda.to_device(data)

expected = np.bincount(data, minlength=NUM_BINS).astype(np.int32)

print("Testing histograms...")
# TODO: Test all three kernels against expected

print("\nBenchmarking...")
# TODO: Time each kernel (50 iterations), compute throughput in GB/s

print("\nComputing CDF from histogram...")
# TODO: Compute cumulative distribution function using np.cumsum
# Print first 10 and last 10 CDF values

## Your tasks

Implement a high-performance privatized histogram that computes the intensity distribution of a simulated grayscale image.

**Requirements:**
1. Implement a naive histogram kernel using global atomic adds (baseline)
2. Implement a privatized histogram kernel using shared memory:
   - Each block maintains a private 256-bin histogram in shared memory
   - Accumulate using shared memory atomic adds
   - Merge per-block histograms into global histogram
3. Implement a coarsened version where each thread processes 4 elements
4. Verify all three produce identical results
5. Benchmark all three approaches and report throughput in GB/s
6. Use the histogram to compute a CDF (cumulative distribution function) via prefix sum

**Hints:**
- For a 256-bin histogram, shared memory uses only 256 * 4 = 1024 bytes -- well within limits
- Initialize shared memory: have the first `num_bins` threads set their bin to 0, then syncthreads()
- For coarsening, compute `base = (blockIdx.x * blockDim.x + threadIdx.x) * coarsen_factor`
- CDF is simply an inclusive scan (cumsum) of the histogram
- Throughput = data_size_in_bytes / time_in_seconds

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-histogram-sort) for the solution and discussion.